# THEMIS v5 — Fine-Tuning Mistral 7B Instruct v0.3

## 🎯 Objective
Fine-tune `unsloth/mistral-7b-instruct-v0.3-bnb-4bit` on Indian statutory law using **52,170 training examples** covering BNS, BNSS, BSA, IPC, RTI, and Constitutional law.

## 📊 Dataset
- **Source**: `danieldeshmukh/themis-legal-training-dataset`
- **Size**: 52,170 examples
- **Components**:
  - Section-based citation lookups (16,514)
  - GSMS-B Legal QA (6,354)
  - IndicLegalQA - Supreme Court judgments (10,002)
  - RTI Cases (1,218)
  - Constitution (870)

## 🏗️ Architecture
- **Base Model**: Mistral 7B Instruct v0.3 (4-bit NF4)
- **Method**: LoRA (Low-Rank Adaptation)
- **Target Modules**: q_proj, k_proj, v_proj, o_proj
- **Rank**: 16, **Alpha**: 32

## ⚡ Hardware
- Kaggle T4 GPU (16GB VRAM)
- ~13GB VRAM usage expected

---
## 📋 Expected Training Loss Reference Table

| Epoch | Step | Expected Loss Range | Status | Action Required |
|-------|------|---------------------|--------|------------------|
| 0.0 | 0 | 2.5 - 3.5 | 🟡 Initializing | Wait for first metrics |
| 0.01 | 100 | 2.0 - 2.8 | 🟢 Normal | Continue training |
| 0.02 | 200 | 1.7 - 2.5 | 🟢 Normal | Continue training |
| 0.05 | 500 | 1.3 - 1.8 | 🟢 Normal | Continue training |
| 0.1 | 1000 | 1.0 - 1.5 | 🟢 Normal | Continue training |
| 0.15 | 1500 | 0.85 - 1.2 | 🟢 Normal | Continue training |
| 0.2 | 2000 | 0.75 - 1.05 | 🟢 Normal | Continue training |
| 0.3 | 3000 | 0.65 - 0.9 | 🟢 Normal | Continue training |
| 0.4 | 4000 | 0.58 - 0.82 | 🟢 Normal | Continue training |
| 0.5 | 5000 | 0.52 - 0.75 | 🟢 Normal | Continue training |
| 0.75 | 7500 | 0.45 - 0.65 | 🟢 Normal | Continue training |
| 1.0 | 10000 | 0.40 - 0.58 | 🟢 Normal | Continue training |
| 1.5 | 15000 | 0.35 - 0.50 | 🟢 Normal | Continue training |
| 2.0 | 20000 | 0.32 - 0.45 | 🟢 Normal | Continue training |
| 2.5 | 25000 | 0.30 - 0.42 | 🟢 Normal | Continue training |
| 3.0 | 30000 | 0.28 - 0.40 | 🟢 Normal | Consider stopping |

### 🔴 Overfitting Indicators (STOP TRAINING)
- Training loss **decreases** but validation loss **increases**
- Training loss drops below **0.25** (too low for legal text)
- Model starts **memorizing** exact phrases instead of learning patterns
- Loss becomes **unstable** (large spikes > 0.5 between steps)
- Gradient norm **explodes** (> 10.0)

### 🟡 Underfitting Indicators (INCREASE TRAINING)
- Training loss **stays above 0.8** after 5000 steps
- Loss **plateaus** (no decrease for 1000+ steps)
- Model outputs are **generic** and don't reference specific sections
- Gradient norm is **too low** (< 0.01)

### 🟢 Healthy Training Signs
- Loss **steadily decreases** with small fluctuations
- Gradient norm stays between **0.1 - 2.0**
- Learning rate follows **cosine schedule** (starts high, decays)
- Loss fluctuations are **< 0.1** between consecutive steps

In [ ]:
# ==================================================
# CELL 1: Install Dependencies
# ==================================================
!pip install -q --upgrade pip
!pip install -q torch==2.4.1 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q unsloth==2024.15.5 @ https://github.com/unslothai/unsloth/archive/main.tar.gz
!pip install -q transformers==4.47.1 datasets accelerate peft==0.14.0 trl==0.17.0 huggingface_hub tensorboard wandb
!pip install -q bitsandbytes==0.45.0

import torch
print(f'\n✅ PyTorch: {torch.__version__}')
print(f'✅ CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
    print(f'✅ VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')

In [ ]:
# ==================================================
# CELL 2: Configuration
# ==================================================
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class TrainingConfig:
    # Model
    base_model: str = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"
    
    # LoRA
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.10
    lora_target_modules: list = field(default_factory=lambda: ["q_proj", "k_proj", "v_proj", "o_proj"])
    
    # Training
    max_seq_length: int = 2048
    num_train_epochs: int = 2
    per_device_train_batch_size: int = 4
    gradient_accumulation_steps: int = 4
    learning_rate: float = 1e-4
    lr_scheduler_type: str = "cosine"
    warmup_steps: int = 100
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    fp16: bool = True
    bf16: bool = False
    
    # Logging & Checkpointing
    logging_steps: int = 10
    save_steps: int = 100
    save_total_limit: int = 5
    
    # Dataset
    dataset_name: str = "danieldeshmukh/themis-legal-training-dataset"
    dataset_file: str = "themis-dataset.json"
    
    # HF Tokens (set via Kaggle secrets or environment variables)
    hf_read_token: str = ""
    hf_write_token: str = ""
    hf_repo: str = "Daniel2503/themis-mistral-7b-lora-v5"
    
    # Resume
    resume_from_checkpoint: Optional[str] = None

config = TrainingConfig()

# Load tokens from environment or Kaggle secrets
import os
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    config.hf_read_token = secrets.get_secret("HF_TOKEN")
    config.hf_write_token = secrets.get_secret("HF_TOKEN")
except Exception:
    config.hf_read_token = os.environ.get("HF_TOKEN", "")
    config.hf_write_token = os.environ.get("HF_TOKEN", "")

print("\n" + "=" * 60)
print("THEMIS v5 Training Configuration")
print("=" * 60)
for k, v in vars(config).items():
    if 'token' not in k.lower():
        print(f"  {k}: {v}")
print("=" * 60)

In [ ]:
# ==================================================
# CELL 3: Load Model with LoRA
# ==================================================
from unsloth import FastLanguageModel
import gc

print("\nLoading model...")
print(f"Base model: {config.base_model}")
print(f"LoRA rank: {config.lora_r}, alpha: {config.lora_alpha}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=config.base_model,
    max_seq_length=config.max_seq_length,
    dtype=None,
    load_in_4bit=True,
    token=config.hf_read_token,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=config.lora_r,
    target_modules=config.lora_target_modules,
    lora_alpha=config.lora_alpha,
    lora_dropout=config.lora_dropout,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

# Print trainable parameters
model.print_trainable_parameters()

# VRAM usage
if torch.cuda.is_available():
    vram_used = torch.cuda.memory_allocated() / 1024**3
    vram_reserved = torch.cuda.memory_reserved() / 1024**3
    print(f"\n📊 VRAM Usage: {vram_used:.2f} GB allocated, {vram_reserved:.2f} GB reserved")

gc.collect()
print("\n✅ Model loaded successfully!")

In [ ]:
# ==================================================
# CELL 4: Load and Prepare Dataset
# ==================================================
from datasets import load_dataset, Dataset
import json
import os

print("\nLoading dataset...")

# Try loading from local file first
dataset_path = config.dataset_file
if os.path.exists(dataset_path):
    print(f"Loading from local file: {dataset_path}")
    with open(dataset_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    dataset = Dataset.from_list(data)
else:
    print(f"Downloading from Kaggle: {config.dataset_name}")
    dataset = load_dataset(config.dataset_name, split="train")

print(f"\n📊 Dataset Statistics:")
print(f"  Total examples: {len(dataset):,}")
print(f"  Sample keys: {list(dataset[0].keys())}")

# Show sample
print(f"\n📝 Sample Example:")
print(f"  Instruction: {dataset[0]['instruction'][:100]}...")
print(f"  Output: {dataset[0]['output'][:100]}...")

# Split: 95% train, 5% validation
split = dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = split['train']
val_dataset = split['test']

print(f"\n📊 Split:")
print(f"  Train: {len(train_dataset):,}")
print(f"  Validation: {len(val_dataset):,}")

# Calculate total steps
total_steps = (
    len(train_dataset) 
    // config.per_device_train_batch_size 
    // config.gradient_accumulation_steps 
    * config.num_train_epochs
)
print(f"\n📈 Training Plan:")
print(f"  Total steps: {total_steps:,}")
print(f"  Steps per epoch: {total_steps // config.num_train_epochs:,}")
print(f"  Checkpoints saved: Every {config.save_steps} steps")
print(f"  Metrics logged: Every {config.logging_steps} steps")

In [ ]:
# ==================================================
# CELL 5: Format Dataset for Mistral Instruct
# ==================================================

def format_mistral(example):
    """Format example for Mistral Instruct v0.3"""
    system_prompt = (
        "You are THEMIS, a legal intelligence engine specializing in Indian statutory law. "
        "Provide accurate, citation-based answers with section references. "
        "Always cite the specific section number and act name. "
        "DISCLAIMER: This is legal orientation, not legal advice."
    )
    
    instruction = example.get('instruction', '')
    inp = example.get('input', '')
    output = example.get('output', '')
    
    if inp:
        user_msg = f"{instruction}\n\nContext: {inp}"
    else:
        user_msg = instruction
    
    # Mistral Instruct format
    text = (
        f"<s>[INST] {system_prompt}\n\n{user_msg} [/INST] {output}</s>"
    )
    
    return {"text": text}

# Apply formatting
train_dataset = train_dataset.map(format_mistral, remove_columns=train_dataset.column_names)
val_dataset = val_dataset.map(format_mistral, remove_columns=val_dataset.column_names)

print("\n✅ Dataset formatted for Mistral Instruct")
print(f"\n📝 Sample formatted text (first 500 chars):")
print(train_dataset[0]['text'][:500])
print("...")

In [ ]:
# ==================================================
# CELL 6: Custom Callbacks for Monitoring
# ==================================================
from transformers import TrainerCallback
import json
import time

class THEMISMonitorCallback(TrainerCallback):
    """Custom callback for detailed training monitoring"""
    
    def __init__(self):
        self.metrics_history = []
        self.start_time = None
        self.last_loss = None
        self.loss_spikes = 0
        self.consecutive_increases = 0
    
    def on_train_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()
        print("\n" + "=" * 70)
        print("🚀 TRAINING STARTED")
        print("=" * 70)
        print(f"  Total steps: {state.max_steps:,}")
        print(f"  Batch size: {args.per_device_train_batch_size}")
        print(f"  Gradient accumulation: {args.gradient_accumulation_steps}")
        print(f"  Effective batch size: {args.per_device_train_batch_size * args.gradient_accumulation_steps}")
        print("=" * 70)
    
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return
        
        step = state.global_step
        loss = logs.get('loss')
        learning_rate = logs.get('learning_rate')
        epoch = logs.get('epoch', 0)
        
        # Store metrics
        metrics = {
            'step': step,
            'loss': loss,
            'learning_rate': learning_rate,
            'epoch': epoch,
            'timestamp': time.time()
        }
        self.metrics_history.append(metrics)
        
        # Loss spike detection
        if self.last_loss is not None and loss is not None:
            if loss - self.last_loss > 0.5:
                self.loss_spikes += 1
                print(f"\n  ⚠️  LOSS SPIKE detected at step {step}! (increase: {loss - self.last_loss:.3f})")
            
            if loss > self.last_loss:
                self.consecutive_increases += 1
            else:
                self.consecutive_increases = 0
            
            if self.consecutive_increases >= 5:
                print(f"\n  ⚠️  WARNING: Loss increased {self.consecutive_increases} times in a row!")
        
        self.last_loss = loss
        
        # Detailed logging every 10 steps
        if step % 10 == 0 and step > 0:
            elapsed = time.time() - self.start_time
            steps_per_sec = step / elapsed if elapsed > 0 else 0
            eta = (state.max_steps - step) / steps_per_sec if steps_per_sec > 0 else 0
            
            # Status indicator
            if loss < 0.3:
                status = "🟢 EXCELLENT"
            elif loss < 0.5:
                status = "🟢 GOOD"
            elif loss < 0.8:
                status = "🟡 ACCEPTABLE"
            elif loss < 1.2:
                status = "🟠 HIGH"
            else:
                status = "🔴 VERY HIGH"
            
            print(f"\n  Step {step:>6,}/{state.max_steps:,} | "
                  f"Loss: {loss:.4f} | "
                  f"LR: {learning_rate:.2e} | "
                  f"Epoch: {epoch:.2f} | "
                  f"{status}")
            print(f"           Speed: {steps_per_sec:.2f} steps/sec | "
                  f"ETA: {eta/60:.1f} min | "
                  f"Spikes: {self.loss_spikes} | "
                  f"Consec. increases: {self.consecutive_increases}")
    
    def on_save(self, args, state, control, **kwargs):
        step = state.global_step
        print(f"\n  💾 CHECKPOINT SAVED at step {step}")
        
        # Save metrics history
        metrics_path = f"metrics_history_step_{step}.json"
        with open(metrics_path, 'w') as f:
            json.dump(self.metrics_history, f, indent=2)
        print(f"  📊 Metrics saved to {metrics_path}")
    
    def on_train_end(self, args, state, control, **kwargs):
        elapsed = time.time() - self.start_time
        print("\n" + "=" * 70)
        print("✅ TRAINING COMPLETED")
        print("=" * 70)
        print(f"  Total time: {elapsed/3600:.2f} hours")
        print(f"  Total steps: {state.global_step:,}")
        print(f"  Final loss: {self.metrics_history[-1]['loss']:.4f}")
        print(f"  Loss spikes: {self.loss_spikes}")
        
        # Final assessment
        final_loss = self.metrics_history[-1]['loss']
        if final_loss < 0.3:
            print("\n  🎯 ASSESSMENT: Model achieved EXCELLENT loss (< 0.3)")
        elif final_loss < 0.5:
            print("\n  🎯 ASSESSMENT: Model achieved GOOD loss (< 0.5)")
        elif final_loss < 0.8:
            print("\n  🎯 ASSESSMENT: Model achieved ACCEPTABLE loss (< 0.8)")
        else:
            print("\n  ⚠️  ASSESSMENT: Loss is still HIGH. Consider more training.")
        print("=" * 70)

print("✅ THEMISMonitorCallback defined")

In [ ]:
# ==================================================
# CELL 7: Configure Trainer
# ==================================================
from trl import SFTTrainer
from transformers import TrainingArguments
from torch.utils.data import DataLoader

# Training arguments
training_args = TrainingArguments(
    output_dir="themis-v5-checkpoints",
    num_train_epochs=config.num_train_epochs,
    per_device_train_batch_size=config.per_device_train_batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    learning_rate=config.learning_rate,
    lr_scheduler_type=config.lr_scheduler_type,
    warmup_steps=config.warmup_steps,
    weight_decay=config.weight_decay,
    max_grad_norm=config.max_grad_norm,
    fp16=config.fp16,
    bf16=config.bf16,
    logging_steps=config.logging_steps,
    save_steps=config.save_steps,
    save_total_limit=config.save_total_limit,
    optim="adamw_8bit",
    seed=3407,
    report_to="none",
    eval_strategy="steps",
    eval_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    dataloader_num_workers=2,
    remove_unused_columns=False,
)

# Initialize callback
monitor_callback = THEMISMonitorCallback()

# Initialize trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
    max_seq_length=config.max_seq_length,
    dataset_text_field="text",
    callbacks=[monitor_callback],
)

print("\n✅ Trainer configured")
print(f"\n📊 Training Configuration:")
print(f"  Total epochs: {config.num_train_epochs}")
print(f"  Batch size: {config.per_device_train_batch_size}")
print(f"  Gradient accumulation: {config.gradient_accumulation_steps}")
print(f"  Effective batch size: {config.per_device_train_batch_size * config.gradient_accumulation_steps}")
print(f"  Learning rate: {config.learning_rate}")
print(f"  LR scheduler: {config.lr_scheduler_type}")
print(f"  Warmup steps: {config.warmup_steps}")
print(f"  Max grad norm: {config.max_grad_norm}")
print(f"  Checkpoint interval: Every {config.save_steps} steps")
print(f"  Logging interval: Every {config.logging_steps} steps")

In [ ]:
# ==================================================
# CELL 8: Pre-Training Validation
# ==================================================
print("\n🔍 Running pre-training validation...")

# Verify tokenizer
test_text = "What does Section 302 of the Bharatiya Nyaya Sanhita say?"
tokens = tokenizer(test_text, return_tensors="pt")
print(f"\n✅ Tokenizer test:")
print(f"  Input: {test_text}")
print(f"  Tokens: {tokens['input_ids'].shape[1]}")

# Verify model loads
print(f"\n✅ Model loaded:")
print(f"  Parameters: {model.num_parameters():,}")
print(f"  Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# VRAM check
if torch.cuda.is_available():
    vram = torch.cuda.memory_allocated() / 1024**3
    print(f"\n✅ VRAM: {vram:.2f} GB")
    if vram > 14:
        print("  ⚠️  WARNING: High VRAM usage. May cause OOM.")
    else:
        print("  🟢 VRAM usage is healthy.")

print("\n" + "=" * 60)
print("✅ Pre-training validation PASSED")
print("=" * 60)

In [ ]:
# ==================================================
# CELL 9: 🚀 START TRAINING
# ==================================================
print("\n" + "=" * 70)
print("🚀 STARTING THEMIS v5 TRAINING")
print("=" * 70)

# Resume from checkpoint if specified
if config.resume_from_checkpoint:
    print(f"\n📂 Resuming from checkpoint: {config.resume_from_checkpoint}")
    trainer_stats = trainer.train(resume_from_checkpoint=config.resume_from_checkpoint)
else:
    print("\n🆕 Starting fresh training...")
    trainer_stats = trainer.train()

print("\n" + "=" * 70)
print("✅ TRAINING COMPLETED")
print("=" * 70)

In [ ]:
# ==================================================
# CELL 10: Training Summary & Metrics
# ==================================================
print("\n" + "=" * 70)
print("📊 TRAINING SUMMARY")
print("=" * 70)

# Basic metrics
print(f"\n📈 Metrics:")
print(f"  Total steps: {trainer_state.global_step:,}")
print(f"  Training loss: {trainer_stats.training_loss:.4f}")
print(f"  Training runtime: {trainer_stats.metrics.get('train_runtime', 0)/3600:.2f} hours")
print(f"  Training samples/sec: {trainer_stats.metrics.get('train_samples_per_second', 0):.2f}")
print(f"  Training steps/sec: {trainer_stats.metrics.get('train_steps_per_second', 0):.2f}")

# Final validation
print(f"\n🔍 Running final validation...")
eval_results = trainer.evaluate()
print(f"  Validation loss: {eval_results['eval_loss']:.4f}")

# Overfitting check
loss_diff = trainer_stats.training_loss - eval_results['eval_loss']
print(f"\n🎯 Overfitting Analysis:")
print(f"  Train loss: {trainer_stats.training_loss:.4f}")
print(f"  Val loss: {eval_results['eval_loss']:.4f}")
print(f"  Difference: {loss_diff:.4f}")

if abs(loss_diff) < 0.1:
    print("  ✅ NO OVERFITTING - Losses are close")
elif loss_diff < -0.1:
    print("  ⚠️  POSSIBLE OVERFITTING - Val loss much higher than train loss")
else:
    print("  ⚠️  UNUSUAL - Train loss much higher than val loss")

print("=" * 70)

In [ ]:
# ==================================================
# CELL 11: Save Model Locally
# ==================================================
import os

local_save_dir = "themis-v5-final"
os.makedirs(local_save_dir, exist_ok=True)

print(f"\n💾 Saving model to {local_save_dir}...")

# Save LoRA adapter
model.save_pretrained(f"{local_save_dir}/adapter")
tokenizer.save_pretrained(f"{local_save_dir}/adapter")
print(f"  ✅ LoRA adapter saved")

# Save merged model (optional - takes more space)
# model.save_pretrained_merged(f"{local_save_dir}/merged", tokenizer, save_method="merged_16bit")
# print(f"  ✅ Merged model saved")

# Save training config
import json
config_dict = vars(config)
# Remove tokens from saved config
config_dict.pop('hf_read_token', None)
config_dict.pop('hf_write_token', None)

with open(f"{local_save_dir}/training_config.json", 'w') as f:
    json.dump(config_dict, f, indent=2)
print(f"  ✅ Training config saved")

# Save metrics history
if monitor_callback.metrics_history:
    with open(f"{local_save_dir}/metrics_history.json", 'w') as f:
        json.dump(monitor_callback.metrics_history, f, indent=2)
    print(f"  ✅ Metrics history saved")

print(f"\n📁 Saved files:")
for root, dirs, files in os.walk(local_save_dir):
    level = root.replace(local_save_dir, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        print(f"{subindent}{file}")

In [ ]:
# ==================================================
# CELL 12: Push to HuggingFace Hub
# ==================================================
print(f"\n📤 Pushing to HuggingFace Hub: {config.hf_repo}")

# Push LoRA adapter
print("\n1. Pushing LoRA adapter...")
model.push_to_hub(
    config.hf_repo,
    token=config.hf_write_token,
    private=True,
)
print(f"  ✅ Adapter pushed to {config.hf_repo}")

# Push tokenizer
print("\n2. Pushing tokenizer...")
tokenizer.push_to_hub(
    config.hf_repo,
    token=config.hf_write_token,
)
print(f"  ✅ Tokenizer pushed")

# Push merged model (optional)
# print("\n3. Pushing merged model...")
# model.push_to_hub_merged(
#     config.hf_repo,
#     tokenizer,
#     save_method="merged_16bit",
#     token=config.hf_write_token,
# )
# print(f"  ✅ Merged model pushed")

print("\n" + "=" * 60)
print("✅ HUGGINGFACE PUSH COMPLETE")
print("=" * 60)

In [ ]:
# ==================================================
# CELL 13: Quick Inference Test
# ==================================================
print("\n🧪 Running inference test...")

# Switch to inference mode
FastLanguageModel.for_inference(model)

test_prompts = [
    "What does Section 302 of the Bharatiya Nyaya Sanhita say about murder?",
    "Explain Section 1 of the Bharatiya Nagarik Suraksha Sanhita.",
    "What are the provisions under Section 63 of the Bharatiya Sakshya Adhiniyam?",
]

for i, prompt in enumerate(test_prompts, 1):
    print(f"\n{'='*60}")
    print(f"Test {i}: {prompt}")
    print("=" * 60)
    
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"\nResponse:\n{response}")

print("\n✅ Inference test complete!")

In [ ]:
# ==================================================
# CELL 14: 🔄 RESUME TRAINING (Run this to resume)
# ==================================================
# Uncomment and modify the checkpoint path to resume training

# from transformers import AutoModelForCausalLM, AutoTokenizer
# from peft import PeftModel
# 
# CHECKPOINT_PATH = "themis-v5-checkpoints/checkpoint-XXXX"  # Replace XXXX with step number
# 
# print(f"\n🔄 Resuming training from: {CHECKPOINT_PATH}")
# 
# # Reload model from checkpoint
# model = AutoModelForCausalLM.from_pretrained(
#     CHECKPOINT_PATH,
#     load_in_4bit=True,
#     token=config.hf_read_token,
# )
# 
# # Reload LoRA weights
# model = PeftModel.from_pretrained(model, f"{CHECKPOINT_PATH}/adapter_model")
# 
# # Reconfigure trainer
# trainer = SFTTrainer(
#     model=model,
#     tokenizer=tokenizer,
#     train_dataset=train_dataset,
#     eval_dataset=val_dataset,
#     args=training_args,
#     max_seq_length=config.max_seq_length,
#     dataset_text_field="text",
#     callbacks=[THEMISMonitorCallback()],
# )
# 
# # Resume training
# trainer.train(resume_from_checkpoint=CHECKPOINT_PATH)
# print("\n✅ Training resumed!")

---
## 📊 Post-Training Analysis

### Expected Final Metrics (v5 Target)

| Metric | Target | Acceptable | Poor |
|--------|--------|------------|------|
| Training Loss | 0.30 - 0.45 | 0.45 - 0.60 | > 0.60 |
| Validation Loss | 0.35 - 0.50 | 0.50 - 0.65 | > 0.65 |
| Loss Difference (train-val) | < 0.05 | 0.05 - 0.15 | > 0.15 |
| Training Time (T4) | 4-6 hours | 6-8 hours | > 8 hours |

### 🔍 How to Detect Overfitting

1. **Loss Divergence**: If val loss starts increasing while train loss keeps decreasing
2. **Memorization**: Model outputs exact training examples instead of generalizing
3. **Gradient Explosion**: Loss spikes > 0.5 between consecutive steps
4. **Unstable Training**: Large oscillations in loss (> 0.3 between steps)

### 🎯 How to Detect Underfitting

1. **High Loss**: Training loss stays above 0.8 after 5000 steps
2. **No Learning**: Loss plateaus for 1000+ steps
3. **Generic Outputs**: Model doesn't reference specific sections
4. **Low Gradient Norm**: Gradients < 0.01 indicate poor learning signal

### 📈 Healthy Training Profile

```
Step    Loss    LR           Status
─────────────────────────────────────────
0       2.8     0.000000     🟡 Initializing
100     2.1     1.00e-05     🟢 Learning
500     1.4     5.00e-05     🟢 Improving
1000    1.0     1.00e-04     🟢 Good
2000    0.8     9.50e-05     🟢 Good
5000    0.55    7.00e-05     🟢 Excellent
10000   0.42    4.00e-05     🟢 Excellent
20000   0.35    1.50e-05     🟢 Excellent
30000   0.32    2.00e-06     🟢 Converged
```

### 🚨 Emergency Actions

| Symptom | Action |
|---------|--------|
| Loss > 5.0 | Stop immediately, check data quality |
| Loss spikes > 1.0 | Reduce learning rate by 50% |
| Val loss increasing | Add early stopping, reduce epochs |
| OOM error | Reduce batch size to 2 |
| Loss plateau | Increase learning rate or add data |

---

## 📚 References

- [Mistral 7B v0.3](https://huggingface.co/mistralai/Mistral-7B-v0.3)
- [Unsloth](https://github.com/unslothai/unsloth)
- [LoRA Paper](https://arxiv.org/abs/2106.09685)
- [THEMIS Project](https://github.com/your-repo/themis)